# TerKet Demo

This notebook shows how to use the repo, how TerKet reports backend choices, and how to run the unified benchmark script.

## Repo Quickstart

From repo root:

```powershell
python -m venv .venv
.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install -e .[dev]
```

To also install benchmark and notebook extras:

```powershell
pip install -r requirements.txt
```

Run test suite:

```powershell
pytest
```

Main paths:

- `src/terket/`: library code
- `benchmarks/run_benchmarks.py`: unified benchmark entrypoint
- `tests/`: smoke and regression tests
- `results/`: generated CSV outputs

In [7]:
from pathlib import Path
import subprocess
import sys

_cwd = Path.cwd()
repo_root = next(p for p in [_cwd, _cwd.parent] if (p / "src" / "terket").exists())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

repo_root

WindowsPath('c:/Users/ethan/github/bee/TerKet')

## Basic Amplitude Query

TerKet accepts either a native `CircuitSpec` or a Qiskit circuit. The simplest exact query asks for one transition amplitude.

In [8]:
from terket import compute_circuit_amplitude, make_circuit

circuit = make_circuit(1, [("h", 0)])
amplitude, info = compute_circuit_amplitude(circuit, [0], [0], as_complex=True)
complex(amplitude), info

((0.7071067811865476+0j),
 {'initial_free': 0,
  'quad_eliminated': 0,
  'constraint_eliminated': 0,
  'branched': 0,
  'remaining_free': 0,
  'branches': 1,
  'cost_model_r': 0,
  'cubic_obstruction': 0,
  'has_cubic_obstruction': False,
  'gauss_obstruction': 0,
  'has_gauss_obstruction': False,
  'phase_states': 0,
  'phase_splits': 0,
  'phase3_backend': None,
  'is_zero': False})

## Backend Metadata

TerKet reports structural diagnostics through `analyze_circuit()` and execution metadata through `compute_*()` helpers.

Backend names you may see in `phase3_backend`:

- `q3_free`: exact q3-free reducer handled circuit without a cubic fallback.
- `treewidth_dp` or `treewidth_dp_peeled`: exact dynamic programming on a low-treewidth cubic core.
- `q3_cover`: exact branch over a bounded cubic support cover.
- `q3_separator`: exact separator-based split of cubic core.
- `cubic_contraction_cpu`: exact contraction-style fallback on cubic residuals.
- `mixed`: different branches used different exact backends.

In [9]:
from terket import analyze_circuit

analysis = analyze_circuit(circuit, [0], [0])
analysis

{'initial_free': 0,
 'quad_eliminated': 0,
 'constraint_eliminated': 0,
 'branched': 0,
 'remaining_free': 0,
 'branches': 1,
 'cost_model_r': 0,
 'cubic_obstruction': 0,
 'has_cubic_obstruction': False,
 'gauss_obstruction': 0,
 'has_gauss_obstruction': False,
 'phase_states': 0,
 'phase_splits': 0,
 'phase3_backend': 'quadratic_tensor',
 'is_zero': False}

## Qiskit Import Path

For repo users working from Qiskit, `from_qiskit()` converts into TerKet's supported basis. This is the normal bridge for imported workloads and benchmark cases.

In [10]:
from qiskit import QuantumCircuit
from terket import from_qiskit

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

spec = from_qiskit(qc)
amp, info = compute_circuit_amplitude(spec, [0, 0], [0, 0], as_complex=True)
complex(amp), info["phase3_backend"]

((0.7071067811865476+0j), None)

## Optional Backends

TerKet can run with optional native and tensor-network dependencies. This cell checks what the local environment exposes.

In [11]:
from terket.backends import _get_quimb_tensor_module, _load_schur_native_module

{
    "native_accelerator_loaded": _load_schur_native_module() is not None,
    "quimb_available": _get_quimb_tensor_module() is not None,
}

{'native_accelerator_loaded': True, 'quimb_available': True}

## MQT Bench Integration

[MQT Bench](https://github.com/cda-tum/mqt-bench) provides a standard library of quantum circuit benchmarks across families like GHZ, BV, QFT, QAOA, and VQE. TerKet accepts MQT Bench circuits directly via `normalize_circuit()`.

Install with `pip install mqt-bench`.

Available families include `ghz`, `graphstate`, `bv`, `dj`, `qft`, `qftentangled`, `qpeexact`, `qaoa`, `vqe_two_local`, `vqe_real_amp`, `qnn`, `randomcircuit`, and others.

In [12]:
try:
    from mqt.bench import get_benchmark_alg
    _mqt_available = True
except ImportError:
    _mqt_available = False
    print("mqt-bench not installed — pip install mqt-bench")

if _mqt_available:
    from terket import normalize_circuit

    # GHZ circuit on 4 qubits creates (|0000> + |1111>) / sqrt(2)
    mqt_circuit = get_benchmark_alg("ghz", circuit_size=4, random_parameters=False)
    mqt_spec = normalize_circuit(mqt_circuit)

    n = mqt_spec.n_qubits
    amp, info = compute_circuit_amplitude(mqt_spec, [0] * n, [1] * n, as_complex=True)
    print(f"n_qubits={n}  amplitude={complex(amp):.6f}  backend={info['phase3_backend']}")
    print(f"n_gates={len(mqt_spec.gates)}")

mqt-bench not installed — pip install mqt-bench


## Unified Benchmark Script

All benchmark families route through one script:

```text
python benchmarks/run_benchmarks.py <family> [family args]
```

Available families:

- `curated`: showcase and fair head-to-head cases in one run, including MQT Bench circuits
- `head-to-head`: TerKet versus quimb amplitude timing on fixed benchmark suites
- `structured-showcase`: structural showcase cases that surface solver diagnostics and residual width
- `depth-scaling`: controlled depth sweeps for TerKet versus quimb
- `amplitude-post-elimination-tensor-rcs`: inspect tensor viability after elimination on RCS residuals
- `rcs-import-strategy-probe`: compare structural impact of alternate RCS import strategies

The `curated` family accepts `--fair-case mqt:<family>:<size>`, `--terket-case mqt:<family>:<size>`, and `--showcase-case <name>` selectors.

In [13]:
result = subprocess.run(
    [
        sys.executable,
        str(repo_root / "benchmarks" / "run_benchmarks.py"),
        "head-to-head",
        "--suite",
        "smoke",
        "--repeats",
        "1",
    ],
    cwd=repo_root,
    text=True,
    capture_output=True,
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

grover16: TerKet=0.001299s, quimb=11.293651s, ratio=8696.789x, abs_error=5.329e-15, backend=treewidth_dp_peeled
qaoa16: TerKet=0.051798s, quimb=0.127212s, ratio=2.456x, abs_error=8.694e-17, backend=q3_free
approximate_qft32: TerKet=0.000628s, quimb=0.143541s, ratio=228.715x, abs_error=3.219e-20, backend=quadratic_tensor
toffoli_ladder16: TerKet=0.000920s, quimb=0.251891s, ratio=273.914x, abs_error=6.328e-15, backend=q3_free
draper8: TerKet=0.001732s, quimb=5.251939s, ratio=3032.648x, abs_error=7.008e-15, backend=q3_free
qec_repetition_magic_round8: TerKet=0.000923s, quimb=0.068842s, ratio=74.585x, abs_error=7.702e-16, backend=q3_free
Wrote 6 row(s) to C:\Users\ethan\github\bee\TerKet\results\quimb_head_to_head.csv



## Reading Benchmark Output

Head-to-head CSVs record both runtime and solver diagnostics:

- `cubic_obstruction` and `gauss_obstruction`: how much hard structure survives exact elimination.
- `cost_model_r`: exponent TerKet's chosen exact backend pays after reductions.
- `terket_phase3_backend`: exact backend family chosen for the hard residual.
- `quimb_over_terket_time_ratio`: quick comparison against quimb when that family includes a quimb baseline.

The `results/` directory is created on demand, so smoke runs from this notebook are safe to repeat.